<a href="https://colab.research.google.com/github/MonaKhadija/Data-Analytics-and-Artificial-Intelligence-Project-DAAI/blob/main/Project2_Khadija_Bassiouni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project 2: Data Cleaning and Preparation**

Student name: Khadija Bassiouni.

**Part 1: Environment Setup & Data Cleaning.**

In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import gc

In [3]:
# Unzipping the files that are loaded into the environment
!unzip "Daily Historical Stock Prices (1970 - 2018).zip"

Archive:  Daily Historical Stock Prices (1970 - 2018).zip
  inflating: historical_stock_prices.csv  
  inflating: historical_stocks.csv   


In [4]:
# Loading datasets
stocks_df = pd.read_csv('historical_stocks.csv')
prices_df = pd.read_csv('historical_stock_prices.csv')

In [5]:
# Quick inspection
print("Stocks shape:", stocks_df.shape)
print("Prices shape:", prices_df.shape)

Stocks shape: (6460, 5)
Prices shape: (20973889, 8)


**Formating Dates and Sorting Data:**

In [6]:
# Formating dates and clean early
prices_df['date'] = pd.to_datetime(prices_df['date'])
prices_df = prices_df[prices_df['ticker'] != 'Symbol']
prices_df = prices_df[(prices_df['close'] > 0) & (prices_df['adj_close'] > 0)]
prices_df = prices_df.sort_values(by=['ticker', 'date']).reset_index(drop=True)

**Part 2 — Create Useful Features :**

In [7]:
# Downcasting numerical prices right away to save RAM
for col in ['open', 'high', 'low', 'close', 'adj_close']:
    prices_df[col] = pd.to_numeric(prices_df[col], downcast='float')
prices_df['volume'] = pd.to_numeric(prices_df['volume'], downcast='integer')

gc.collect()

print("Calculating features...")
prices_df['price_change'] = prices_df.groupby('ticker')['close'].diff()
prices_df['daily_return'] = prices_df.groupby('ticker')['close'].pct_change()
prices_df['close_lag_1'] = prices_df.groupby('ticker')['close'].shift(1)
prices_df['ma_5'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.rolling(window=5).mean())
prices_df['ma_20'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.rolling(window=20).mean())
prices_df['volatility_20'] = prices_df.groupby('ticker')['daily_return'].transform(lambda x: x.rolling(window=20).std())
prices_df['high_low_range'] = prices_df['high'] - prices_df['low']

Calculating features...


**Part 3 — Combine and Organize the Data:**

In [8]:
# Downcast new features
for col in ['price_change', 'daily_return', 'close_lag_1', 'ma_5', 'ma_20', 'volatility_20', 'high_low_range']:
    prices_df[col] = pd.to_numeric(prices_df[col], downcast='float')

gc.collect()

0

In [9]:
# Merging and immediately deleting prices_df to free memory
merged_df = pd.merge(prices_df, stocks_df[['ticker', 'exchange', 'sector', 'industry']], on='ticker', how='left')
del prices_df
del stocks_df
gc.collect()

print("Dropping NaNs...")
cleaned_model_df = merged_df.dropna(subset=['ma_20', 'volatility_20'])
del merged_df
gc.collect()

print("Success! Cleaned shape:", cleaned_model_df.shape)

Dropping NaNs...
Success! Cleaned shape: (20860334, 18)


In [10]:
#  one-hot encoding  of Categorical Variables
import numpy as np
import gc

print("Performing safe one-hot encoding...")

# Converting categorical variables into binary columns using 1-byte integers (uint8) to save RAM
cleaned_model_df = pd.get_dummies(
    cleaned_model_df,
    columns=['exchange', 'sector'],
    drop_first=True,
    dtype=np.uint8
)

gc.collect()
print("One-hot encoding successful! New shape:", cleaned_model_df.shape)

Performing safe one-hot encoding...
One-hot encoding successful! New shape: (20860334, 28)


**Part 4 — Prepare the Final Dataset:**

In [11]:
#  Selecting the Final Features
final_cols = [
    'ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume',
    'price_change', 'daily_return', 'close_lag_1', 'ma_5', 'ma_20',
    'volatility_20', 'high_low_range', 'industry'
]

# Dynamically grabing the one-hot encoded exchange and sector columns we created earlier
encoded_cols = [col for col in cleaned_model_df.columns if col.startswith('exchange_') or col.startswith('sector_')]

# Creating the final dataset dataframe
final_dataset = cleaned_model_df[final_cols + encoded_cols].copy()

In [12]:
#  Checking the Final Dataset (Data Integrity Checks)
print("--- Final Dataset Quality Checks ---")
print(f"Total Rows: {final_dataset.shape[0]:,}")
print(f"Total Columns: {final_dataset.shape[1]}")
print(f"Missing Values Count: {final_dataset.isnull().sum().sum()}")
print(f"Duplicate (Ticker + Date) Records: {final_dataset.duplicated(subset=['ticker', 'date']).sum()}")
print(f"Date Range: {final_dataset['date'].min()} to {final_dataset['date'].max()}")
print(f"Data Types Sample:\n{final_dataset.dtypes.head(10)}")

--- Final Dataset Quality Checks ---
Total Rows: 20,860,334
Total Columns: 28
Missing Values Count: 2531217
Duplicate (Ticker + Date) Records: 0
Date Range: 1970-01-30 00:00:00 to 2018-08-24 00:00:00
Data Types Sample:
ticker                  object
date            datetime64[ns]
open                   float32
high                   float32
low                    float32
close                  float32
adj_close              float64
volume                   int64
price_change           float32
daily_return           float32
dtype: object


In [13]:
#  Saving the Prepared Dataset
final_dataset.to_csv('stock_prepared.csv', index=False)